# Simulation Data Loading for Machine Learning

이 노트북은 MATLAB에서 생성한 HDF5 형식의 시뮬레이션 데이터를 PyTorch로 로드하고 머신러닝 모델에 사용하는 방법을 보여줍니다.

## 목차
1. 라이브러리 임포트
2. 데이터 로드 및 확인
3. DataLoader 설정
4. 데이터 통계 분석
5. 간단한 모델 예제
6. 훈련 루프 예제

## 1. 라이브러리 임포트

In [1]:
import sys
import os

# Add pytorch folder to path
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import h5py
import matplotlib.pyplot as plt
from dataset import SimulationDataset, SimulationDatasets
from load_h5_data import load_h5_simulation_data, get_dataset_info

# GPU 사용 가능 여부 확인
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

Using device: cuda
PyTorch version: 2.9.1+cu128


## 2. 데이터 로드 및 확인

### 2.1 HDF5 파일 정보 확인

In [2]:
# 데이터 파일 경로
h5_path = '../data/simulation_data.h5'

# 파일 존재 여부 확인
if not os.path.exists(h5_path):
    print(f"⚠️ 파일을 찾을 수 없습니다: {h5_path}")
    print("MATLAB에서 preSimulateH5()를 실행해주세요.")
else:
    print(f"✓ 파일 발견: {h5_path}")
    file_size_mb = os.path.getsize(h5_path) / (1024**2)
    print(f"파일 크기: {file_size_mb:.2f} MB")

✓ 파일 발견: ../data/simulation_data.h5
파일 크기: 169.38 MB


### 2.2 데이터 상세 정보

In [6]:
# 데이터 정보 출력
get_dataset_info(h5_path)

Detailed Dataset Information:

Dataset: P0
  Shape: (5, 2, 2)
  dtype: float64
  Size: 20 elements
  Memory usage: 0.00 MB


AttributeError: 'Dataset' object has no attribute 'flat'

## 3. DataLoader 설정

### 3.1 SimulationDataset 생성 (PyTorch Dataset)

In [ ]:
# 데이터셋 생성
dataset = SimulationDataset(h5_filepath=h5_path)

print(f"✓ Dataset 생성 완료")
print(f"  - 총 샘플 수: {len(dataset)}")
print(f"  - 노이즈 레벨: {dataset.num_noise_levels}")
print(f"  - 이터레이션: {dataset.num_iterations}")
print(f"  - 포인트: {dataset.num_points}")

### 3.2 샘플 확인

In [ ]:
# 첫 번째 샘플 확인
sample = dataset[0]

print("Sample keys:")
for key, value in sample.items():
    if torch.is_tensor(value):
        print(f"  {key}: shape={value.shape}, dtype={value.dtype}")
    else:
        print(f"  {key}: {value}")

### 3.3 DataLoader 생성

In [ ]:
# 훈련/검증 데이터 분할
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

from torch.utils.data import random_split
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# DataLoader 생성
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"✓ DataLoader 생성 완료")
print(f"  - 훈련 샘플: {train_size}")
print(f"  - 검증 샘플: {val_size}")
print(f"  - 배치 크기: {batch_size}")
print(f"  - 훈련 배치 수: {len(train_loader)}")
print(f"  - 검증 배치 수: {len(val_loader)}")

### 3.4 배치 데이터 확인

In [ ]:
# 첫 배치 확인
batch = next(iter(train_loader))

print("Batch shapes:")
for key, value in batch.items():
    if torch.is_tensor(value):
        print(f"  {key}: {value.shape}")
    else:
        print(f"  {key}: {value}")

## 4. 데이터 통계 분석

In [ ]:
# 전체 데이터셋의 통계
datasets = SimulationDatasets(h5_filepath=h5_path)
datasets.printinfo()

### 4.1 데이터 시각화

In [ ]:
# toaPos 데이터 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 모든 노이즈 레벨의 toaPos 시각화
for noise_idx in range(datasets.num_noise_levels):
    data = datasets.get_dataset_for_noise_level(noise_idx)
    # 평균 위치
    mean_pos = data['toaPos'].mean(dim=(0, 1))
    axes[0].scatter(mean_pos[:, 0], mean_pos[:, 1], label=f'Noise level {noise_idx}')

axes[0].set_xlabel('X')
axes[0].set_ylabel('Y')
axes[0].set_title('Mean Position by Noise Level')
axes[0].legend()
axes[0].grid(True)

# processbias 통계
processbias = datasets.processbias
axes[1].bar(range(datasets.num_noise_levels), processbias[0, :], alpha=0.7, label='X bias')
axes[1].bar(range(datasets.num_noise_levels), processbias[1, :], alpha=0.7, label='Y bias')
axes[1].set_xlabel('Noise Level')
axes[1].set_ylabel('Bias')
axes[1].set_title('Process Bias by Noise Level')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 5. 간단한 모델 예제

### 5.1 위치 예측 모델
입력: z (거리 차이 측정), 출력: toaPos (예상 위치)

In [ ]:
class PositionPredictor(nn.Module):
    """위치를 예측하는 간단한 신경망 모델"""
    def __init__(self, input_dim=6, hidden_dim=64, output_dim=2):
        super(PositionPredictor, self).__init__()
        
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, z):
        return self.net(z)

# 모델 생성
model = PositionPredictor(input_dim=6, hidden_dim=64, output_dim=2)
model = model.to(device)

print("✓ Model created")
print(model)

# 파라미터 수
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

### 5.2 손실 함수 및 옵티마이저

In [ ]:
# 손실 함수 및 옵티마이저
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print("✓ Loss function: MSELoss")
print("✓ Optimizer: Adam")
print(f"  - Initial LR: 0.001")
print("✓ Scheduler: StepLR")

## 6. 훈련 루프 예제

### 6.1 훈련 함수

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """한 에포크의 훈련"""
    model.train()
    total_loss = 0.0
    num_batches = 0
    
    for batch in dataloader:
        # 입출력 데이터 준비
        z = batch['z'].to(device)
        toaPos = batch['toaPos'].to(device)
        
        # Forward pass
        pred_pos = model(z)
        loss = criterion(pred_pos, toaPos)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
    
    return total_loss / num_batches

def validate_epoch(model, dataloader, criterion, device):
    """검증"""
    model.eval()
    total_loss = 0.0
    num_batches = 0
    
    with torch.no_grad():
        for batch in dataloader:
            z = batch['z'].to(device)
            toaPos = batch['toaPos'].to(device)
            
            pred_pos = model(z)
            loss = criterion(pred_pos, toaPos)
            
            total_loss += loss.item()
            num_batches += 1
    
    return total_loss / num_batches

print("✓ 훈련 및 검증 함수 정의 완료")

### 6.2 모델 훈련

In [ ]:
# 훈련 하이퍼파라미터
num_epochs = 10
early_stopping_patience = 3

# 훈련 이력
train_losses = []
val_losses = []
best_val_loss = float('inf')
patience_counter = 0

print(f"시작 훈련: {num_epochs} epochs")
print("=" * 60)

for epoch in range(num_epochs):
    # 훈련
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(train_loss)
    
    # 검증
    val_loss = validate_epoch(model, val_loader, criterion, device)
    val_losses.append(val_loss)
    
    # Learning rate 스케줄링
    scheduler.step()
    
    # 조기 종료
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
    else:
        patience_counter += 1
    
    # 진행 상황 출력
    print(f"Epoch {epoch+1:2d}/{num_epochs} | "
          f"Train Loss: {train_loss:.6f} | "
          f"Val Loss: {val_loss:.6f} | "
          f"LR: {optimizer.param_groups[0]['lr']:.6f}")
    
    if patience_counter >= early_stopping_patience:
        print(f"조기 종료! (Patience: {patience_counter}/{early_stopping_patience})")
        break

print("=" * 60)
print(f"훈련 완료! 최고 검증 손실: {best_val_loss:.6f}")

### 6.3 손실 함수 시각화

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss', marker='o')
plt.plot(val_losses, label='Validation Loss', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('Training History')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

### 6.4 예측 결과 분석

In [ ]:
# 검증 데이터에 대한 예측
model.eval()
predictions = []
ground_truth = []
errors = []

with torch.no_grad():
    for batch in val_loader:
        z = batch['z'].to(device)
        toaPos = batch['toaPos'].to(device)
        
        pred = model(z).cpu().numpy()
        gt = toaPos.cpu().numpy()
        
        predictions.append(pred)
        ground_truth.append(gt)
        errors.append(np.linalg.norm(pred - gt, axis=1))

predictions = np.concatenate(predictions)
ground_truth = np.concatenate(ground_truth)
errors = np.concatenate(errors)

print("예측 성능 통계:")
print(f"  평균 오차 (MAE): {np.mean(np.abs(predictions - ground_truth)):.6f}")
print(f"  RMSE: {np.sqrt(np.mean((predictions - ground_truth)**2)):.6f}")
print(f"  평균 거리 오차: {np.mean(errors):.6f}")
print(f"  최대 오차: {np.max(errors):.6f}")
print(f"  표준편차: {np.std(errors):.6f}")

### 6.5 예측 결과 시각화

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 예측 vs 실제값 - X 좌표
axes[0, 0].scatter(ground_truth[:, 0], predictions[:, 0], alpha=0.5)
axes[0, 0].plot([ground_truth[:, 0].min(), ground_truth[:, 0].max()],
                 [ground_truth[:, 0].min(), ground_truth[:, 0].max()],
                 'r--', lw=2, label='Perfect prediction')
axes[0, 0].set_xlabel('Ground Truth (X)')
axes[0, 0].set_ylabel('Prediction (X)')
axes[0, 0].set_title('X Coordinate Prediction')
axes[0, 0].legend()
axes[0, 0].grid(True)

# 예측 vs 실제값 - Y 좌표
axes[0, 1].scatter(ground_truth[:, 1], predictions[:, 1], alpha=0.5)
axes[0, 1].plot([ground_truth[:, 1].min(), ground_truth[:, 1].max()],
                 [ground_truth[:, 1].min(), ground_truth[:, 1].max()],
                 'r--', lw=2, label='Perfect prediction')
axes[0, 1].set_xlabel('Ground Truth (Y)')
axes[0, 1].set_ylabel('Prediction (Y)')
axes[0, 1].set_title('Y Coordinate Prediction')
axes[0, 1].legend()
axes[0, 1].grid(True)

# 오차 분포
axes[1, 0].hist(errors, bins=30, edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Distance Error')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Prediction Error Distribution')
axes[1, 0].grid(True)

# 위치 시각화 (첫 100개 샘플)
axes[1, 1].scatter(ground_truth[:100, 0], ground_truth[:100, 1], 
                   label='Ground Truth', alpha=0.6, s=50)
axes[1, 1].scatter(predictions[:100, 0], predictions[:100, 1], 
                   label='Prediction', alpha=0.6, s=50, marker='x')
axes[1, 1].set_xlabel('X')
axes[1, 1].set_ylabel('Y')
axes[1, 1].set_title('Position Predictions (First 100 samples)')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

## 7. 모델 저장 및 로드

In [ ]:
# 모델 저장
model_path = 'position_predictor_model.pth'
torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'best_val_loss': best_val_loss,
}, model_path)

print(f"✓ 모델 저장됨: {model_path}")

# 모델 로드 예제
# checkpoint = torch.load(model_path)
# model.load_state_dict(checkpoint['model_state_dict'])
# optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
# print(f"✓ 모델 로드됨")

## 8. 노이즈 레벨별 분석

In [ ]:
# 노이즈 레벨별로 데이터 로드
noise_names = ['0.01', '0.1', '1', '10', '100']

fig, axes = plt.subplots(1, datasets.num_noise_levels, figsize=(15, 3))

for noise_idx in range(datasets.num_noise_levels):
    data = datasets.get_dataset_for_noise_level(noise_idx)
    
    # 평균 위치
    mean_pos = data['toaPos'].mean(dim=(0, 1)).numpy()
    
    # 산점도
    axes[noise_idx].scatter(data['toaPos'][:, :, 0].numpy().flatten(), 
                            data['toaPos'][:, :, 1].numpy().flatten(),
                            alpha=0.3, s=10)
    axes[noise_idx].scatter(mean_pos[0], mean_pos[1], 
                           color='red', s=200, marker='*', 
                           edgecolors='black', linewidth=2)
    axes[noise_idx].set_xlabel('X')
    axes[noise_idx].set_ylabel('Y')
    axes[noise_idx].set_title(f'Noise Level: {noise_names[noise_idx]}')
    axes[noise_idx].grid(True)

plt.tight_layout()
plt.show()

## 요약

이 노트북에서 다룬 내용:

1. ✅ HDF5 데이터 로드 및 확인
2. ✅ PyTorch DataLoader 설정
3. ✅ 데이터 시각화 및 통계 분석
4. ✅ 신경망 모델 구성
5. ✅ 훈련 루프 구현
6. ✅ 예측 성능 평가
7. ✅ 모델 저장 및 로드
8. ✅ 노이즈 레벨별 분석

**다음 단계:**
- 더 복잡한 모델 구현
- 교차 검증 또는 K-Fold 평가
- Hyperparameter 튜닝
- 노이즈 레벨별 모델 학습